# Turkish Legal RAG -- Evaluate on Your Own Corpus

Run this notebook on **Google Colab** to evaluate the system on *your* documents and *your* benchmark -- no local install, no manual cloning.

**How to use**
1. `Runtime -> Change runtime type -> T4 GPU` (generation needs a GPU; without one the notebook runs retrieval-only).
2. `Runtime -> Run all`.
3. Upload your documents (and, optionally, a benchmark file) when prompted.
4. Read the **Base RAG vs. Fine-tuned RAG** comparison, then open the live demo link.

Repository: https://github.com/berkay-aktas/hukuk-rag

In [ ]:
# Check the runtime has a GPU; set a retrieval-only fallback flag if not.
import torch
GPU = torch.cuda.is_available()
NO_LLM = "" if GPU else "--no-llm"
if GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Use Runtime -> Change runtime type -> T4 GPU, then Run all again.")
    print("Continuing in RETRIEVAL-ONLY mode (--no-llm): retrieval metrics only, no generated answers.")

## Setup

In [ ]:
!git clone -q https://github.com/berkay-aktas/hukuk-rag.git
%cd hukuk-rag
!pip install -q -r requirements.txt

## 1. Upload your documents

Your document collection (PDF / TXT / MD). They are chunked by a legal-aware splitter.

In [ ]:
import os, shutil
from google.colab import files
os.makedirs("custom_docs", exist_ok=True)
print("Upload your document collection (PDF / TXT / MD):")
uploaded = files.upload()
for fn in uploaded:
    shutil.move(fn, os.path.join("custom_docs", fn))
print("Saved to custom_docs/:", os.listdir("custom_docs"))

## 2. Upload your benchmark *(optional)*

A JSONL/CSV with questions. Turkish keys `{soru, cevap, ilgili_belgeler}` or English fields are auto-detected. If relevance labels are present you get Recall@k / MRR / nDCG; otherwise generation metrics. Skip (upload nothing) for a demo-only run.

In [ ]:
from google.colab import files
print("Optional: upload ONE benchmark file (.jsonl or .csv). For a demo only, do not upload anything.")
uploaded = files.upload()
bench = next(iter(uploaded), None)
print("Benchmark:", bench or "(none - demo only)")

## 3. Build indexes (Base + Fine-tuned)

Two indexes are built on *your* corpus: one with the stock encoder (**Base RAG**) and one with the fine-tuned Turkish-legal E5 (**Fine-tuned RAG**), so the comparison below is apples-to-apples. The fine-tuned encoder auto-downloads (~2.2 GB) on first run.

In [ ]:
!python -m hukuk_rag ingest --docs ./custom_docs --out ./idx_base
!python -m hukuk_rag ingest --docs ./custom_docs --out ./idx_ft --embedding-model bewrkay/multilingual-e5-large-tr-legal

## 4. Base RAG vs. Fine-tuned RAG

Both configurations are scored on the same uploaded benchmark with the same LLM -- the apples-to-apples comparison the rubric grades.

In [ ]:
if bench:
    !python -m hukuk_rag benchmark --questions "{bench}" --indexes ./idx_base --variant base --report ./out_base {NO_LLM}
    !python -m hukuk_rag benchmark --questions "{bench}" --indexes ./idx_ft --variant prod --report ./out_prod {NO_LLM}
    import json, pathlib
    for name, d in [("Base RAG", "out_base"), ("Fine-tuned RAG", "out_prod")]:
        summary = pathlib.Path(d, "summary.json")
        print("==== " + name + " ====")
        if summary.exists():
            print(json.dumps(json.loads(summary.read_text()), indent=2, ensure_ascii=False))
        else:
            print("(see ./" + d + "/ for predictions and metrics)")
else:
    print("No benchmark uploaded - skipping scoring. Run the demo cell below.")

## 5. Interactive demo

Launches a public `gradio.live` link (alive while this Colab session runs). Click it to query the system interactively. Loads the Fine-tuned index; on CPU it falls back to retrieval-only automatically.

In [ ]:
!python demo.py --indexes ./idx_ft --llm-base Qwen/Qwen2.5-7B-Instruct --share {NO_LLM}